# Carla 서버 제어
### 1. carla.Client
Carla : 서버-클라이언트 구조를 기반으로 한 오픈 소스 자율 주행 시뮬레이터.

carla.Client(host=127.0.0.1, port=2000) : 서버의 world를 조정하거나 객체를 생성하고 다루기 위한 클라이언트 객체.

    - host : Carla 시뮬레이터 서버가 작동하고 있는 host의 IP.
    - port : Calra 시뮬레이터 서버가 작동하고 있는 host의 TCP 포트.



In [3]:
import carla
client = carla.Client('localhost', 2000)


### 2. carla.Client()를 이용한 시뮬레이션 환경 설정
<!-- carla.Client().get_world() : 현재 Carla 시뮬레이터 서버의 맵, 날씨, Actor 등에 접근할 수 있는 객체를 반환. -->
carla.Client().get_available_maps() : 현재 시뮬레이터 서버에서 사용 가능한 맵 리스트 반환.
carla.Client().load_world('맵 이름') : 현재 시뮬레이터 서버의 맵을 변경.

In [4]:
client.get_available_maps()


['/Game/Carla/Maps/Town05_Opt',
 '/Game/Carla/Maps/Town10HD',
 '/Game/Carla/Maps/Town04',
 '/Game/Carla/Maps/Town03_Opt',
 '/Game/Carla/Maps/Town04_Opt',
 '/Game/Carla/Maps/Town02_Opt',
 '/Game/Carla/Maps/Town03',
 '/Game/Carla/Maps/Town02',
 '/Game/Carla/Maps/Town01_Opt',
 '/Game/Carla/Maps/Town05',
 '/Game/Carla/Maps/Town10HD_Opt',
 '/Game/Carla/Maps/Town01']

In [5]:
client.load_world('Town04')


In [6]:
world = client.get_world()
# world.set_weather(carla.WeatherParameters.MidRainSunset)

In [5]:
for vehicle in world.get_blueprint_library().filter('vehicle.*'):
    print(vehicle.id)
    # print(type(vehicle))

vehicle.audi.a2
vehicle.citroen.c3
vehicle.chevrolet.impala
vehicle.dodge.charger_police_2020
vehicle.micro.microlino
vehicle.dodge.charger_police
vehicle.audi.tt
vehicle.jeep.wrangler_rubicon
vehicle.mercedes.coupe
vehicle.mercedes.coupe_2020
vehicle.harley-davidson.low_rider
vehicle.dodge.charger_2020
vehicle.ford.ambulance
vehicle.lincoln.mkz_2020
vehicle.mini.cooper_s_2021
vehicle.toyota.prius
vehicle.ford.crown
vehicle.carlamotors.carlacola
vehicle.vespa.zx125
vehicle.nissan.patrol_2021
vehicle.mercedes.sprinter
vehicle.audi.etron
vehicle.seat.leon
vehicle.volkswagen.t2_2021
vehicle.tesla.cybertruck
vehicle.lincoln.mkz_2017
vehicle.ford.mustang
vehicle.carlamotors.firetruck
vehicle.volkswagen.t2
vehicle.mitsubishi.fusorosa
vehicle.tesla.model3
vehicle.diamondback.century
vehicle.gazelle.omafiets
vehicle.bmw.grandtourer
vehicle.bh.crossbike
vehicle.kawasaki.ninja
vehicle.nissan.patrol
vehicle.nissan.micra
vehicle.mini.cooper_s
vehicle.yamaha.yzf


In [6]:
vehicle_bp = world.get_blueprint_library().filter('vehicle.dodge.charger_police')
print(type(vehicle_bp)) # class 'carla.libcarla.BlueprintLibrary'
pos = carla.Transform(carla.Location(x=210, y=195, z=10), carla.Rotation(pitch=0, yaw=180, roll=0)) # spawn 위치 직접 설
actor1 = world.spawn_actor(vehicle_bp[0], pos) # param1: blueprint, param2: transform

<class 'carla.libcarla.BlueprintLibrary'>


In [7]:
# 충돌이 있는 경우 Actor는 생성되지 않습니다.
# 이러한 충돌을 방지하고 안전하게 생성하고 싶다면, 추천 스폰 지점 목록을 활용할 수 있습니다.
spawn_points = world.get_map().get_spawn_points()
actor2 = world.spawn_actor(vehicle_bp[0], spawn_points[0])

In [8]:
# 이러한 액터는 자동으로 사라지지 않습니다.
# 액터는 명시적으로 제거해야 합니다.
actor1.destroy()
actor2.destroy()


True

In [9]:
# actor의 id나 식별자를 알 수 없을 경우 world.get_actors()로 전체 액터를 가져와서 확인할 수 있습니다.
actors = world.get_actors()
# for a in actors:
    # print(a.destroy()) # 전체 액터 제거
print(actors)
print(actors[0].destroy()) # 전체 액터 중 첫 번째 액터 제거


[Actor(id=149, type=vehicle.dodge.charger_police), Actor(id=148, type=vehicle.dodge.charger_police), Actor(id=147, type=vehicle.dodge.charger_police), Actor(id=146, type=traffic.traffic_light), Actor(id=145, type=traffic.traffic_light), Actor(id=144, type=traffic.traffic_light), Actor(id=143, type=traffic.traffic_light), Actor(id=142, type=traffic.traffic_light), Actor(id=141, type=traffic.traffic_light), Actor(id=140, type=traffic.traffic_light), Actor(id=139, type=traffic.traffic_light), Actor(id=138, type=traffic.traffic_light), Actor(id=137, type=traffic.traffic_light), Actor(id=136, type=traffic.traffic_light), Actor(id=135, type=traffic.traffic_light), Actor(id=134, type=traffic.traffic_light), Actor(id=133, type=traffic.traffic_light), Actor(id=132, type=traffic.traffic_light), Actor(id=131, type=traffic.traffic_light), Actor(id=130, type=traffic.traffic_light), Actor(id=129, type=traffic.traffic_light), Actor(id=128, type=traffic.traffic_light), Actor(id=127, type=traffic.traff

# 센서 데이터 및 수집

carla.Sensor 클래스는 데이터를 측정하고 스트리밍할 수 있는 액터 클래스 입니다.
데이터 유형은 carla.SensorData이며 listen() 함수를 통해 데이터를 수신 및 관리합니다.



In [ ]:
# 다양한 Actor의 유형

In [8]:
# 카메라 센서 설정
camera_bp = world.get_blueprint_library().filter('sensor.camera.rgb')[0] # blueprint 라이브러리에서 카메라 센서의 blueprint를 가져오기
# 카메라 센서의 경우 image_size_x, image_size_y, 시야각 등의 속성을 설정할 수 있습니다.
camera_bp.set_attribute('image_size_x', '1920')
camera_bp.set_attribute('image_size_y', '1080')
camera_bp.set_attribute('fov', '110')

camera_bp.set_attribute('sensor_tick', '5.0') # 센서가 데이터를 생성하는 간격을 설정하는 속성입니다. 5.0으로 설정하면 센서가 5초마다 데이터를 생성합니다.

# 카메라 센서 부착
camera_transform = carla.Transform(carla.Location(x=0, y=0, z=2), carla.Rotation(pitch=0, yaw=0, roll=0)) # 카메라의 위치와 회전을 정의하는 Transform 객체 생성
camera = world.spawn_actor(camera_bp, camera_transform, attach_to=actor1, attachment_type=carla.AttachmentType.Rigid) # 카메라 센서를 actor1에 부착. 카메라의 transform은 부착할 actor에 기반한 위치. attach_type은 Rigid로 설정하여 카메라가 차량과 완전히 고정되도록 합니다.



In [ ]:
# 센서 데이터 수집 
# 파라미터는 반드시 람다 콜백함수여야 한다.
# camera.listen(lambda image: image.save_to_disk('output/%06d.png' % image.frame)) # 카메라 센서가 캡처한 이미지를 저장하는 콜백 함수 등록

camera.stop() # 카메라 센서의 데이터 수집 중지

In [ ]:
# 시멘틱 세그멘테이션 카메라 센서 설정
semantic_bp = world.get_blueprint_library().filter('sensor.camera.semantic_segmentation')[0]
semantic_bp.set_attribute('image_size_x', '1920')
semantic_bp.set_attribute('image_size_y', '1080')
semantic_bp.set_attribute('fov', '110')
semantic_bp.set_attribute('sensor_tick', '5.0')

semantic_transform = carla.Transform(carla.Location(x=0, y=0, z=2), carla.Rotation(pitch=0, yaw=0, roll=0))
semantic_camera = world.spawn_actor(semantic_bp, camera_transform, attach_to=actor1, attachment_type=carla.AttachmentType.Rigid)




In [31]:
# 시멘틱 세그멘테이션 카메라의 데이터 수집
# 시멘틱 세그멘테이션 카메라의 경우  carla.ColorConverter의 CityScapesPalette를 활용하여 시멘틱 세그멘테이션 이미지에 색상 맵을 적용하여 저장합니다.
# semantic_camera.listen(lambda image: image.save_to_disk('output/semantic/%06d.png' % image.timestamp, carla.ColorConverter.CityScapesPalette))


# 시멘틱 세그멘테이션 카메라의 데이터 수집 중지
semantic_camera.stop() 




In [29]:
# 이외에도 라이더, 레이더, GNSS, IMU 등 다양한 센서가 존재합니다. 각 센서의 blueprint를 가져와서 원하는 속성을 설정하고, 차량에 부착하여 데이터를 수집할 수 있습니다.
sensor_bp  = world.get_blueprint_library().filter('sensor.*')
for sensor in sensor_bp:
    print(sensor.id)



sensor.other.lane_invasion
sensor.other.collision
sensor.camera.depth
sensor.lidar.ray_cast
sensor.camera.semantic_segmentation
sensor.other.radar
sensor.camera.instance_segmentation
sensor.camera.rgb
sensor.other.rss
sensor.camera.optical_flow
sensor.lidar.ray_cast_semantic
sensor.other.obstacle
sensor.other.imu
sensor.other.gnss
sensor.camera.dvs
sensor.camera.normals


# CARLA에서의 교통 시뮬레이션
### 1. Traffic Manager

Carla의 Traffic Manager는 시뮬레이션 내 다양한 Actor들이 자율적으로 행동하도록 제어하는 모듈입니다.
이때 Actor는 카메라나 라이다와 같은 센서 기반 자율이 아니라 내부적으로 수집하는 Actor들의 위치, 속도, 상태 등을 기반으로 작동합니다.

-- 나중에 시간 나면 추가하기. 일단 내가 한 것들 부터 완전히 해놓고 생각하자.


# 차량의 생성 및 제어
3.1 blueprint 로딩 및 role_name='hero' 설정
CARLA에서 role_name=hero는 중요하다. CARLA에서 ego_vehicle을 지정하는 속성이기 때문이다.
뿐만 아니라 hero 차량을 중심으로 특정 반경의 물리 연산이 집중되며 CARLA에서 제공하는 녹화 기능 혹은 이벤트 로깅 기능 또한 hero 차량을 중심으로 수행되기 때문이다.
따라서 주요 차량에 대하여 블루 프린트 생성 단계에 명시적으로 role_name='hero' 설정을 주입해야 한다.


In [9]:
car1_bp = world.get_blueprint_library().find('vehicle.audi.tt')
car1_bp.set_attribute('role_name', 'hero') 

# 사전 제공되는 안전한 spawn 위치를 활용하여 차량을 생성합니다.
spawn_points_1 = world.get_map().get_spawn_points()



In [10]:
# 혹은 현재 서버 화면(Spectator)의 위치를 활용하여 spawn 위치를 추출합니다.
spectator = world.get_spectator()
spectator_transform = spectator.get_transform()

print(spectator_transform)

spawn_points_2 = spectator_transform
car_1 = world.spawn_actor(car1_bp, spawn_points_2)

Transform(Location(x=138.848526, y=-171.748428, z=2.900956), Rotation(pitch=0.000000, yaw=1.000204, roll=0.000000))


In [11]:
# 혹은 대략적인 X, Y 좌표만 알고 있을 경우 waypoint를 활용하여 차선 중앙 좌표를 추출하고 z축을 살짝 띄워 생성할 수 있습니다.
approximate_location = carla.Location(x=230, y=160)
spawn_points_3 = world.get_map().get_waypoint(approximate_location).transform
spawn_points_3.location.z += 2.0 # z축을 2.0만큼 띄워서 차량이 지면과 충돌하지 않도록 합니다.
print(spawn_points_3)

car_3 = world.spawn_actor(car1_bp, spawn_points_3)

Transform(Location(x=239.022720, y=180.823807, z=2.000000), Rotation(pitch=0.000000, yaw=-23.129292, roll=0.000000))


# 차량 제어하기 : Vehicle Controller
차량 제어의 핵심은 Vehicle Controller의 생성과 설정, 그리고 이를 차량에 적용하는 것입니다.

### 차량 조명
차량의 조명을 제어하기 위해선 carla.VehicleLightState 클래스와 Vehicle.set_light_state() 함수를 사용합니다.
차량에 따라 지원되는 조명의 종류가 다양하므로 참고 바랍니다.


In [12]:
# 좌측 깜빡이를 3초간 키는 상황 예시
import time

l_blinker = carla.VehicleLightState.LeftBlinker
off = carla.VehicleLightState.NONE
car_3.set_light_state(carla.VehicleLightState(l_blinker))
time.sleep(3)
car_3.set_light_state(carla.VehicleLightState(off))

### 차량 움직임
차량의 움직임을 제어하기 위해 carla.VehicleControl 클래스와 Vehicle.apply_control() 함수를 사용합니다.
VehicleControl 클래스엔 이동과 관련된 속성이 조재하며, throttle은 차량의 속력을, steer는 방향과 관련된 속성입니다. 이외에 brake, gear 등의 속성이 존재합니다.




In [14]:
vc = carla.VehicleControl()
vc.throttle = 0 # 가속 페달을 50% 밟는 상황 예시
vc.reverse = True
car_3.apply_control(vc)

In [ ]:
# 시나리오의 구현은 이러한 VehicleController를 적용하는 것의 반복으로 구성됩니다.
# 그러나 이러한 방법은 차량의 쓰로틀 값과 스티어 값을 직접 적용해야하므로 까다롭고 제한적입니다.

# 예시 코드 파일 - Situation_Assessment.ipynb



carla.libcarla.VehicleFailureState.NONE

# Waypoint 기반 차량 주행
Vehicle Controller를 직접 작성하여 조작하는 방식의 한계를 극복하고, 더 정교한 시나리오를 구하기 위해 Waypoint 기반 제어 방법을 사용하였습니다.

Waypoint 기반 제어의 핵심은 CARLA가 제공하는 Waypoint와 PID 컨트롤러를 활용하여 차량이 차선의 중심을 안정적으로 따라가도록 자동 제어하는 방식입니다. 이 방식은 다음의 스텝을 따릅니다.
- 경로 탐색 : 현재 차량 위치에서 가장 가까운 도로 위의 Waypoint를 찾습니다.
- 목표 설정 : 탐색한 Waypoint로부터 일정 거리 앞에 있는 다음 Waypoint를 주행 목표로 설정합니다.
- 자동 제어 : CARLA의 PID 컨트롤러가 목표 Waypoint까지 도달하기 위한 최적의 throttle, brake steer 값을 계산합니다.
- 주행 : PID 컨트롤러가 계산한 제어 값을 차량에 적용하여 목표 지점으로 주행합니다.
- 반복 : 위 과정을 지속 반복하여 경로를 이탈하지 않고 안정적인 주행을 유지합니다.

In [14]:
import carla
import sys

sys.path.append('/home/oem/workspace/carla_workshop/carla_0.9.14/PythonAPI/carla/')
import math

from agents.navigation import controller as ctr
from queue import Queue


class DriveWaypoint(object):
    """
    waypoint 기반 주행 구현.
    """

    def __init__(self, Vehicle, PIDctr, world):
        """
        :param Vehicle: 완성된 carla.Vehicle 객체를 받는 파라미터.
        :param PIDctr: 완성된 carla의 PID Controller 객체를 받는 파라미터.
        :param world: carla.world 객체를 받는 파라미터.
        """
        self.dir_wp_que = Queue()
        self.car = Vehicle
        self.pid = PIDctr
        self.world = world

    def drive(self, debug=False, loop=20000, speed=80):
        current_waypoint = self.world.get_map().get_waypoint(self.car.get_location())

        for _ in range(loop):
            if debug:
                self.world.debug.draw_point(current_waypoint.transform.location, size=0.3)

            if not self.dir_wp_que.empty():
                next_wp = self.dir_wp_que.get()
                vehicle_controller = self.pid.run_step(target_speed=speed, waypoint=next_wp)
                self.car.apply_control(vehicle_controller)
                current_waypoint = next_wp
                print('wp_queue 사용됨')
                print(f'queue empty? : {self.dir_wp_que.empty()}')
            elif self.dir_wp_que.empty():
                next_wp = current_waypoint.next(distance=3)  # distance 3으로 다음 waypoint 생성
                next_wp = next_wp[0]
                c_loc = self.car.get_location()
                x = c_loc.x
                y = c_loc.y
                wp_x = next_wp.transform.location.x
                wp_y = next_wp.transform.location.y
                dist = math.sqrt((x - wp_x) ** 2 + (y - wp_y) ** 2)
                if dist <= 1:
                    current_waypoint = next_wp  # 현재 waypoint 갱신
                    print('waypoint updated')
                else:
                    vehicle_controller = self.pid.run_step(target_speed=speed, waypoint=next_wp)
                    self.car.apply_control(vehicle_controller)

    def add_left_waypoint(self, distance=20):
        current_waypoint = self.world.get_map().get_waypoint(self.car.get_location())
        left_waypoint = current_waypoint.next(distance)[0].get_left_lane()
        self.dir_wp_que.put(left_waypoint)

    def add_right_waypoint(self, distance=20):
        current_waypoint = self.world.get_map().get_waypoint(self.car.get_location())
        right_waypoint = current_waypoint.next(distance)[0].get_right_lane()
        print(right_waypoint)
        self.dir_wp_que.put(right_waypoint)

In [17]:

args_lateral_dict = {'K_P': 1.95, 'K_I': 0.05, 'K_D': 0.2, 'dt': 0.05}
args_long_dict = {'K_P': 1.0, 'K_I': 0.05, 'K_D': 0.0, 'dt': 0.05}


DriveWaypoint(car_3, ctr.VehiclePIDController(car_3, args_lateral_dict, args_long_dict), world).drive(debug=True,loop=100000)

In [9]:
import carla
import math
import sys
sys.path.append('/home/oem/workspace/carla_workshop/carla_0.9.14/PythonAPI/carla/')
from agents.navigation import controller as ctr
from queue import Queue

class DriveWaypoint(object):
    def __init__(self, Vehicle, PIDctr, world):
        self.dir_wp_que = Queue()
        self.car = Vehicle
        self.pid = PIDctr
        self.world = world
        self.current_waypoint = self.world.get_map().get_waypoint(self.car.get_location())

    def drive_step(self, debug=False, speed=40):
        """
        루프 내부의 1틱(Tick)에 해당하는 동작만 수행하도록 변경
        """
        if debug:
            # life_time을 0.1로 주어 렌더링 부하 방지
            self.world.debug.draw_point(self.current_waypoint.transform.location, size=0.1, life_time=0.1)

        if not self.dir_wp_que.empty():
            next_wp = self.dir_wp_que.get()
            vehicle_controller = self.pid.run_step(target_speed=speed, waypoint=next_wp)
            self.car.apply_control(vehicle_controller)
            self.current_waypoint = next_wp
        else:
            # 현재 위치에서 3미터 앞의 웨이포인트 획득
            next_wp = self.current_waypoint.next(distance=3.0)[0]
            
            # 내장 API를 활용한 정확한 3D 거리 계산
            dist = self.car.get_location().distance(next_wp.transform.location)
            
            if dist <= 1.0:
                self.current_waypoint = next_wp # 웨이포인트 도달 시 갱신
            else:
                vehicle_controller = self.pid.run_step(target_speed=speed, waypoint=next_wp)
                self.car.apply_control(vehicle_controller)

# ==========================================================
# 최적화된 메인 실행부
# ==========================================================
args_lateral_dict = {'K_P': 1.95, 'K_I': 0.05, 'K_D': 0.2, 'dt': 0.05}
args_long_dict = {'K_P': 1.0, 'K_I': 0.05, 'K_D': 0.0, 'dt': 0.05}

# 1. 제어기 및 주행 객체는 루프 진입 전 "단 한 번만" 초기화
pid_controller = ctr.VehiclePIDController(car_3, args_lateral_dict, args_long_dict)
driver = DriveWaypoint(car_3, pid_controller, world)

# 2. 메인 스레드에서 무한 루프 실행
try:
    while True:
        world.tick() # 서버 프레임 동기화 (필수)
        driver.drive_step(debug=True, speed=30) # 속도를 80에서 30으로 하향 권장 (커브 안정성)
except KeyboardInterrupt:
    print("주행이 강제 종료되었습니다.")

주행이 강제 종료되었습니다.


In [ ]:
import carla
import sys
sys.path.append('/home/oem/workspace/carla_workshop/carla_0.9.14/PythonAPI/carla/')
from agents.navigation import controller as ctr
from queue import Queue

import math

class DriveWaypoint(object):
    """
    상태 머신(State Machine) 기반의 안전한 Waypoint 추종 클래스
    """
    def __init__(self, vehicle, pid_controller, world):
        self.car = vehicle
        self.pid = pid_controller
        self.world = world
        self.dir_wp_que = Queue()
        
        # '현재 위치'가 아닌 추종해야 할 '목표 웨이포인트'를 별도로 관리
        self.target_waypoint = self.world.get_map().get_waypoint(self.car.get_location())

    def drive_step(self, speed=30.0, debug=False):
        current_loc = self.car.get_location()
        
        # [핵심 수정 1] 차량의 현재 실제 속도(m/s)를 계산
        velocity = self.car.get_velocity()
        current_speed_ms = math.sqrt(velocity.x**2 + velocity.y**2 + velocity.z**2)
        
        # [핵심 수정 2] 속도에 비례하는 동적 주시 거리(Look-ahead) 및 갱신 반경 계산
        # 시속 0일때 최소 3m, 속도가 빠를수록 거리가 멀어짐 (예: 시속 72km(20m/s)일 때 약 13m 앞을 봄)
        dynamic_lookahead = max(3.0, current_speed_ms * 0.5 + 3.0) 
        
        # 목표 도달 판정 거리도 속도에 비례해 넓혀줌 (고속일수록 목표점을 넉넉히 스쳐지나가도 도달한 것으로 인정)
        reach_threshold = max(1.5, current_speed_ms * 0.2)

        dist_to_target = current_loc.distance(self.target_waypoint.transform.location)

        if dist_to_target <= reach_threshold: # 목표점과의 거리가 도달
            if not self.dir_wp_que.empty():
                self.target_waypoint = self.dir_wp_que.get()
                print("[시스템] 큐(Queue)에서 새 이벤트 웨이포인트 적용됨")
            else:
                # [핵심 수정 3] 하드코딩된 3.0 대신 동적 거리 적용
                next_wps = self.target_waypoint.next(dynamic_lookahead)
                if next_wps: 
                    self.target_waypoint = next_wps[0]

        if debug:
            self.world.debug.draw_point(
                self.target_waypoint.transform.location, 
                size=0.15, 
                color=carla.Color(255, 0, 0), # 고속 주행 시 목표점이 멀리 찍히는 것을 빨간 점으로 확인
                life_time=0.1
            )

        control = self.pid.run_step(target_speed=speed, waypoint=self.target_waypoint)
        self.car.apply_control(control)

    def add_left_waypoint(self, distance=20):
        """
        현재 위치에서 distance(m) 앞의 좌측 차선 웨이포인트를 큐에 추가
        """
        current_wp = self.world.get_map().get_waypoint(self.car.get_location())
        next_wps = current_wp.next(distance)
        
        if next_wps:
            left_wp = next_wps[0].get_left_lane()
            # [핵심 방어 로직] 좌측 차선이 실제로 존재하는지 검증
            if left_wp is not None:
                self.dir_wp_que.put(left_wp)
                print(f"[명령] {distance}m 앞 좌측 차선 변경 예약 완료")
            else:
                print("[경고] 좌측에 유효한 차선이 없습니다. 차선 변경 명령 무시됨.")

    def add_right_waypoint(self, distance=20):
        """
        현재 위치에서 distance(m) 앞의 우측 차선 웨이포인트를 큐에 추가
        """
        current_wp = self.world.get_map().get_waypoint(self.car.get_location())
        next_wps = current_wp.next(distance)
        
        if next_wps:
            right_wp = next_wps[0].get_right_lane()
            # [핵심 방어 로직] 우측 차선이 실제로 존재하는지 검증
            if right_wp is not None:
                self.dir_wp_que.put(right_wp)
                print(f"[명령] {distance}m 앞 우측 차선 변경 예약 완료")
            else:
                print("[경고] 우측에 유효한 차선이 없습니다. 차선 변경 명령 무시됨.")

In [23]:
import time

def run_lane_change_scenario():
    # 1. 클라이언트 연결 및 동기 모드 강제 설정 (커널 크래시 방지)
    client = carla.Client('localhost', 2000)
    client.set_timeout(10.0)
    world = client.get_world()
    
    settings = world.get_settings()
    settings.synchronous_mode = True
    settings.fixed_delta_seconds = 0.02 # 50 FPS 고정
    world.apply_settings(settings)

    actor_list = []

    try:
        # 2. Ego Vehicle 스폰
        blueprint_library = world.get_blueprint_library()
        vehicle_bp = blueprint_library.filter('vehicle.audi.tt')[0]
        vehicle_bp.set_attribute('role_name', 'hero')
        
        # spawn_point = world.get_map().get_spawn_points()[0]
        approximate_location = carla.Location(x=230, y=160)
        spawn_point = world.get_map().get_waypoint(approximate_location).transform
        spawn_point.location.z += 0.5 # 물리 엔진 충돌 방지용 Z축 보정
        ego_vehicle = world.spawn_actor(vehicle_bp, spawn_point)
        actor_list.append(ego_vehicle)


        print("\n[시스템] 차량 스폰 완료. 물리 엔진 안착을 위해 1초 대기합니다...")

        brake_control = carla.VehicleControl(throttle=0.0, steer=0.0, brake=1.0)
        ego_vehicle.apply_control(brake_control)

        for _ in range(20): # 20틱 * 0.05초 = 0.5초 대기
            world.tick()
            time.sleep(0.05)
        
        print(f"[시스템] 차량 스폰 완료. 시나리오를 시작합니다.")

        # 3. 제어기 및 DriveWaypoint 객체 단일 초기화
        # args_lateral_dict = {'K_P': 1.95, 'K_I': 0.05, 'K_D': 0.2, 'dt': 0.05}
        args_lateral_dict = {'K_P': 0.8, 'K_I': 0.05, 'K_D': 0.1, 'dt': 0.05} # 고속 주행 시 안정성을 위한 튜닝
        args_long_dict = {'K_P': 1.0, 'K_I': 0.05, 'K_D': 0.0, 'dt': 0.05}
        
        pid_controller = ctr.VehiclePIDController(ego_vehicle, args_lateral_dict, args_long_dict)
        driver = DriveWaypoint(ego_vehicle, pid_controller, world)

        # 4. 메인 제어 루프 (약 10초 주행 = 500 틱 * 0.02초)
        for tick in range(500):
            world.tick() # 서버 프레임 동기화
            time.sleep(0.02)
            
            # [시나리오 트리거] 특정 틱에 도달 시 차선 변경 명령 큐(Queue)에 주입
            if tick == 100: # 2초 경과 시점
                print(f"\n[Tick {tick}] 이벤트 발생: 우측 차선 변경 명령 하달")
                driver.add_right_waypoint(distance=20)
            elif tick == 350: # 7초 경과 시점
                print(f"\n[Tick {tick}] 이벤트 발생: 좌측 차선 변경 (복귀) 명령 하달")
                driver.add_left_waypoint(distance=20)
            
            # 단일 스텝 주행 실행 (디버그 라인 렌더링 활성화)
            driver.drive_step(speed=140.0, debug=True)

    finally:
        # 5. 환경 초기화 (비정상 종료 시에도 반드시 실행되어야 하는 방어 로직)
        print("\n[시스템] 시나리오 종료. 할당된 Actor 및 환경을 초기화합니다.")
        settings = world.get_settings()
        settings.synchronous_mode = False
        world.apply_settings(settings)
        client.apply_batch([carla.command.DestroyActor(x) for x in actor_list])
        print("[시스템] 정리 완료.")

# 코드 실행
run_lane_change_scenario()


[시스템] 차량 스폰 완료. 물리 엔진 안착을 위해 1초 대기합니다...
[시스템] 차량 스폰 완료. 시나리오를 시작합니다.

[Tick 100] 이벤트 발생: 우측 차선 변경 명령 하달
[명령] 20m 앞 우측 차선 변경 예약 완료
[시스템] 큐(Queue)에서 새 이벤트 웨이포인트 적용됨

[Tick 350] 이벤트 발생: 좌측 차선 변경 (복귀) 명령 하달
[명령] 20m 앞 좌측 차선 변경 예약 완료
[시스템] 큐(Queue)에서 새 이벤트 웨이포인트 적용됨

[시스템] 시나리오 종료. 할당된 Actor 및 환경을 초기화합니다.
[시스템] 정리 완료.


In [9]:
client = carla.Client('localhost', 2000)
client.set_timeout(10.0)
world = client.get_world()


In [ ]:

settings = world.get_settings()
settings.synchronous_mode = True
settings.fixed_delta_seconds = 0.05 # 20 FPS 고정

In [ ]:

settings.synchronous_mode = True
settings.fixed_delta_seconds = 0.1 # 20 FPS 고정
world.apply_settings(settings)

158227

: 

In [ ]:
import typing
import collections

# Python 3.7의 typing 모듈에 없는 OrderedDict를 강제로 연결
if not hasattr(typing, 'OrderedDict'):
    typing.OrderedDict = collections.OrderedDict

from ultralytics import YOLO
# model = YOLO('/home/oem/workspace/carla_workshop/YOLO/yolov8n.pt') # base 모델
# results = model.train(data='/home/oem/workspace/carla_workshop/YOLO/coco8.yaml', epochs=100, imgsz=640)

model = YOLO('/home/oem/workspace/carla_workshop/YOLO/runs/detect/train2/weights/best.pt') # coco8 학습 모델
for i in range(1,12):
    results = model.predict(source=f'/home/oem/workspace/carla_workshop/YOLO/test_data/{i}.png', save=True, conf=0.4)

for r in results:
    print(r.boxes)

# waypoint + yolo8 도전해보기
###  목표 1. 카메라 센서를 붙이고 카메라 센서로부터 이미지를 받아와서 YOLO 모델에 넣어서 객체 인식 결과를 얻기
### 목표 2. 객체 인식 결과를 활용하여 차량 주변의 보행자나 다른 차량을 인식하고, 이를 기반으로 급제동 등 상황 연출하기

In [ ]:
# 목표 1 

# carla 라이브러리 임포트 및 클라이언트 연결, 맵 로드
import carla
client = carla.Client('localhost', 2000)
client.load_world('Town04')




Python 3.7.0


In [ ]:
# 차량 actor blueprint 정의

# ego vehicle
car1_bp = world.get_blueprint_library().find('vehicle.mercedes.coupe_2020')
car1_bp.set_attribute('role_name', 'hero')
car1_bp.set_attribute('color','255,255,255')

# 주행-좌
car2_bp = world.get_blueprint_library().find('vehicle.mercedes.coupe_2020')
car2_bp.set_attribute('role_name', 'hero')

# 주행-우
car3_bp = world.get_blueprint_library().find('vehicle.mercedes.coupe_2020')
car3_bp.set_attribute('role_name', 'hero')

# 주행-전방
car4_bp = world.get_blueprint_library().find('vehicle.mercedes.coupe_2020')
car4_bp.set_attribute('role_name', 'hero')

# 장애물 1
car5_bp = world.get_blueprint_library().find('vehicle.citroen.c3')
car5_bp.set_attribute('role_name', 'hero')

# 장애물 2
car6_bp = world.get_blueprint_library().find('vehicle.chevrolet.impala')
car6_bp.set_attribute('role_name', 'hero')

# AMB
car7_bp = world.get_blueprint_library().find('vehicle.ford.ambulance')
car7_bp.set_attribute('role_name', 'hero')

# 차량 소환 위치 정의
spawn_point_1 = carla.Transform(carla.Location(x=175.377335, y=14.680666, z=9.333585), carla.Rotation(pitch=0, yaw=-179.57, roll=0)) # ego
spawn_point_2 = carla.Transform(carla.Location(x=168.311737, y=18.066513, z=9.648239), carla.Rotation(pitch=0, yaw=-179.57, roll=0)) # ego- 좌
spawn_point_3 = carla.Transform(carla.Location(x=165.756546, y=11.099593, z=9.763625), carla.Rotation(pitch=0, yaw=-179.57, roll=0)) # 주행 - 우
spawn_point_4 = carla.Transform(carla.Location(x=160.377335, y=14.680666, z=9.333585), carla.Rotation(pitch=0, yaw=-179.57, roll=0)) # 주행 - 전방
spawn_point_5 = carla.Transform(carla.Location(x=-54.930981, y=13.533400, z=11.055361), carla.Rotation(pitch=-0.808291, yaw=-144.808273, roll=-0.576660)) # 장애물 1
spawn_point_6 = carla.Transform(carla.Location(x=-53.498806, y=10.286665, z=11.085323), carla.Rotation(pitch=-0.896940, yaw=159.454361, roll=0.357310)) # 장애물 2
spawn_point_7 = carla.Transform(carla.Location(x=-48.693542, y=16.434429, z=11.161311), carla.Rotation(pitch=-1.049438, yaw=-179.041122, roll=-0.013855)) # 앰뷸런스


In [ ]:
vc_rt = carla.VehicleControl(reverse=True)
vc_rf = carla.VehicleControl(reverse=False)

# 차량 생성

ego = world.spawn_actor(car1_bp, spawn_point_1)

time.sleep(1)
ego.apply_control(vc_rt)
time.sleep(0.5)
ego.apply_control(vc_rf)

left = world.spawn_actor(car2_bp, spawn_point_2)

time.sleep(1)
left.apply_control(vc_rt)
time.sleep(0.5)
left.apply_control(vc_rf)

right = world.spawn_actor(car3_bp, spawn_point_3)

time.sleep(1)
right.apply_control(vc_rt)
time.sleep(0.5)
right.apply_control(vc_rf)

front = world.spawn_actor(car4_bp, spawn_point_4)

time.sleep(1)
front.apply_control(vc_rt)
time.sleep(0.5)
front.apply_control(vc_rf)

obs_1 = world.spawn_actor(car5_bp, spawn_point_5)

time.sleep(1)
obs_1.apply_control(vc_rt)
time.sleep(0.5)
obs_1.apply_control(vc_rf)

obs_2 = world.spawn_actor(car6_bp, spawn_point_6)

time.sleep(1)
obs_2.apply_control(vc_rt)
time.sleep(0.5)
obs_2.apply_control(vc_rf)

amb = world.spawn_actor(car7_bp, spawn_point_7)

time.sleep(1)
amb.apply_control(vc_rt)
time.sleep(0.5)
amb.apply_control(vc_rf)

In [ ]:
# actor 제거 
actor_list = [ego, left, right, front, obs_1, obs_2, amb]
for actor in actor_list:
    actor.destroy()

In [ ]:
# 시나리오 연출은 어떻게 하면 좋을까..
# 가장 전방의 차량이 주행 하다가 앞에 사고 상황 발견 => 다른 차량에게도 사고 상황 전파 => 다른 차량들도 사고 상황 인지 후 감속 및 정차

In [ ]:
# 카메라 센서 설정
camera_bp = world.get_blueprint_library().filter('sensor.camera.rgb')[0]
camera_bp.set_attribute('image_size_x', '640') # yolo 모델의 최적 입력 크기에 맞춰 설정
camera_bp.set_attribute('image_size_y', '640')
camera_bp.set_attribute('fov', '90')

# 카메라 센서 위치 설정
camera_transform = carla.Transform(carla.Location(x=1.5, z=2.4))

# 카메라 생성 및 ego 차량에 부착
camera = world.spawn_actor(camera_bp, camera_transform, attach_to=front)



In [ ]:


import carla
import sys
sys.path.append('/home/oem/workspace/carla_workshop/carla_0.9.14/PythonAPI/carla/')
from agents.navigation import controller as ctr
from queue import Queue

import math

class DriveWaypoint(object):
    """
    상태 머신(State Machine) 기반의 안전한 Waypoint 추종 클래스
    """
    def __init__(self, vehicle, pid_controller, world):
        self.car = vehicle
        self.pid = pid_controller
        self.world = world
        self.dir_wp_que = Queue()
        self.is_stopping = False # 차량의 현재 주행 상태를 관리하는 플래그
        
        # '현재 위치'가 아닌 추종해야 할 '목표 웨이포인트'를 별도로 관리
        self.target_waypoint = self.world.get_map().get_waypoint(self.car.get_location())

    def drive_step(self, speed=30.0, debug=False):

        if self.is_stopping:
            control = carla.VehicleControl(throttle=0.0, steer=0.0, brake=1.0)
            self.car.apply_control(control)
            return
        
        current_loc = self.car.get_location()
        
        # [핵심 수정 1] 차량의 현재 실제 속도(m/s)를 계산
        velocity = self.car.get_velocity()
        current_speed_ms = math.sqrt(velocity.x**2 + velocity.y**2 + velocity.z**2)
        
        # [핵심 수정 2] 속도에 비례하는 동적 주시 거리(Look-ahead) 및 갱신 반경 계산
        # 시속 0일때 최소 3m, 속도가 빠를수록 거리가 멀어짐 (예: 시속 72km(20m/s)일 때 약 13m 앞을 봄)
        dynamic_lookahead = max(3.0, current_speed_ms * 0.5 + 3.0) 
        
        # 목표 도달 판정 거리도 속도에 비례해 넓혀줌 (고속일수록 목표점을 넉넉히 스쳐지나가도 도달한 것으로 인정)
        reach_threshold = max(1.5, current_speed_ms * 0.2)

        dist_to_target = current_loc.distance(self.target_waypoint.transform.location)

        if dist_to_target <= reach_threshold: # 목표점과의 거리가 도달
            if not self.dir_wp_que.empty():
                self.target_waypoint = self.dir_wp_que.get()
                print("[시스템] 큐(Queue)에서 새 이벤트 웨이포인트 적용됨")
            else:
                # [핵심 수정 3] 하드코딩된 3.0 대신 동적 거리 적용
                next_wps = self.target_waypoint.next(dynamic_lookahead)
                if next_wps: 
                    self.target_waypoint = next_wps[0]

        if debug:
            self.world.debug.draw_point(
                self.target_waypoint.transform.location, 
                size=0.15, 
                color=carla.Color(255, 0, 0), # 고속 주행 시 목표점이 멀리 찍히는 것을 빨간 점으로 확인
                life_time=0.1
            )

        control = self.pid.run_step(target_speed=speed, waypoint=self.target_waypoint)
        self.car.apply_control(control)

    def add_left_waypoint(self, distance=20):
        """
        현재 위치에서 distance(m) 앞의 좌측 차선 웨이포인트를 큐에 추가
        """
        current_wp = self.world.get_map().get_waypoint(self.car.get_location())
        next_wps = current_wp.next(distance)
        
        if next_wps:
            left_wp = next_wps[0].get_left_lane()
            # [핵심 방어 로직] 좌측 차선이 실제로 존재하는지 검증
            if left_wp is not None:
                self.dir_wp_que.put(left_wp)
                print(f"[명령] {distance}m 앞 좌측 차선 변경 예약 완료")
            else:
                print("[경고] 좌측에 유효한 차선이 없습니다. 차선 변경 명령 무시됨.")

    def add_right_waypoint(self, distance=20):
        """
        현재 위치에서 distance(m) 앞의 우측 차선 웨이포인트를 큐에 추가
        """
        current_wp = self.world.get_map().get_waypoint(self.car.get_location())
        next_wps = current_wp.next(distance)
        
        if next_wps:
            right_wp = next_wps[0].get_right_lane()
            # [핵심 방어 로직] 우측 차선이 실제로 존재하는지 검증
            if right_wp is not None:
                self.dir_wp_que.put(right_wp)
                print(f"[명령] {distance}m 앞 우측 차선 변경 예약 완료")
            else:
                print("[경고] 우측에 유효한 차선이 없습니다. 차선 변경 명령 무시됨.")

    def stop(self):
        self.is_stopping = True
        print("[명령] 차량 정지 명령이 적용되었습니다.")
    
    def resume(self):
        self.is_stopping = False
        print("[명령] 차량 주행 재개 명령이 적용되었습니다.")  
        

In [ ]:
import time

def run_lane_change_scenario():
    # 1. 클라이언트 연결 및 동기 모드 강제 설정 (커널 크래시 방지)
    client = carla.Client('localhost', 2000)
    client.set_timeout(10.0)
    world = client.get_world()
    
    settings = world.get_settings()
    settings.synchronous_mode = True
    settings.fixed_delta_seconds = 0.02 # 50 FPS 고정
    world.apply_settings(settings)

    actor_list = []

    try:
        # 2. Ego Vehicle 스폰
        blueprint_library = world.get_blueprint_library()
        vehicle_bp = blueprint_library.filter('vehicle.audi.tt')[0]
        vehicle_bp.set_attribute('role_name', 'hero')
        
        # spawn_point = world.get_map().get_spawn_points()[0]
        approximate_location = carla.Location(x=230, y=160)
        spawn_point = world.get_map().get_waypoint(approximate_location).transform
        spawn_point.location.z += 0.5 # 물리 엔진 충돌 방지용 Z축 보정
        ego_vehicle = world.spawn_actor(vehicle_bp, spawn_point)
        actor_list.append(ego_vehicle)


        print("\n[시스템] 차량 스폰 완료. 물리 엔진 안착을 위해 1초 대기합니다...")

        brake_control = carla.VehicleControl(throttle=0.0, steer=0.0, brake=1.0)
        ego_vehicle.apply_control(brake_control)

        for _ in range(20): # 20틱 * 0.05초 = 0.5초 대기
            world.tick()
            time.sleep(0.05)
        
        print(f"[시스템] 차량 스폰 완료. 시나리오를 시작합니다.")

        # 3. 제어기 및 DriveWaypoint 객체 단일 초기화
        # args_lateral_dict = {'K_P': 1.95, 'K_I': 0.05, 'K_D': 0.2, 'dt': 0.05}
        args_lateral_dict = {'K_P': 0.8, 'K_I': 0.05, 'K_D': 0.1, 'dt': 0.05} # 고속 주행 시 안정성을 위한 튜닝
        args_long_dict = {'K_P': 1.0, 'K_I': 0.05, 'K_D': 0.0, 'dt': 0.05}
        
        pid_controller = ctr.VehiclePIDController(ego_vehicle, args_lateral_dict, args_long_dict)
        driver = DriveWaypoint(ego_vehicle, pid_controller, world)

        # 4. 메인 제어 루프 (약 10초 주행 = 500 틱 * 0.02초)
        for tick in range(500):
            world.tick() # 서버 프레임 동기화
            time.sleep(0.02)
            
            # [시나리오 트리거] 특정 틱에 도달 시 차선 변경 명령 큐(Queue)에 주입
            if tick == 100: # 2초 경과 시점
                print(f"\n[Tick {tick}] 이벤트 발생: 우측 차선 변경 명령 하달")
                driver.add_right_waypoint(distance=20)
            elif tick == 350: # 7초 경과 시점
                print(f"\n[Tick {tick}] 이벤트 발생: 좌측 차선 변경 (복귀) 명령 하달")
                driver.add_left_waypoint(distance=20)
            
            # 단일 스텝 주행 실행 (디버그 라인 렌더링 활성화)
            driver.drive_step(speed=140.0, debug=True)

    finally:
        # 5. 환경 초기화 (비정상 종료 시에도 반드시 실행되어야 하는 방어 로직)
        print("\n[시스템] 시나리오 종료. 할당된 Actor 및 환경을 초기화합니다.")
        settings = world.get_settings()
        settings.synchronous_mode = False
        world.apply_settings(settings)
        client.apply_batch([carla.command.DestroyActor(x) for x in actor_list])
        print("[시스템] 정리 완료.")

# 코드 실행
run_lane_change_scenario()